Source:
https://www.datacamp.com/tutorial/knowledge-graph-rag
https://www.npmjs.com/package/download-git-repo

## PROCESS

1. Load files from repo
2. Create function chunking from each file
    Metadata:
        filename: ?
        line_number_mapping: ?
        function:
        function_call_stack
    Content:
3. Chunking
    TextSplitter
    https://python.langchain.com/docs/how_to/code_splitter/

STEP 2: Initialize language model
1. instantiate language model (OpenAI)
2. llm.transformer.convert_to_graph_documents

STEP 3: Store to vector database
Embeddings

STEP 4: Retrieve knowledge for RAG

Step 5: Evaluate response (langchain)

In [6]:
! pip install --upgrade pip

  Using cached pip-24.3.1-py3-none-any.whl.metadata (3.7 kB)
Using cached pip-24.3.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2


In [20]:
! pip install GitPython
! pip install langchain 
! pip install langchain-community
! pip install langchain_pinecone 
! pip install langchain-text-splitters
! pip install openai
! pip install os
! pip install pinecone
! pip install pinecone-client
! pip install pprint
! pip install pygithub 
! pip install python-dotenv
! pip install requests
! pip install streamlit
! pip install tiktoken
! pip install tree-sitter
! pip install tree-sitter-language-pack


  Using cached langchain_pinecone-0.0.1-py3-none-any.whl.metadata (1.5 kB)
INFO: pip is looking at multiple versions of langchain-pinecone to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following versions that require a different python version: 0.0.2 Requires-Python >=3.8.1,<3.13; 0.0.2rc0 Requires-Python >=3.8.1,<3.13; 0.0.3 Requires-Python >=3.8.1,<3.13; 0.1.0 Requires-Python <3.13,>=3.8.1; 0.1.1 Requires-Python <3.13,>=3.8.1; 0.1.2 Requires-Python <3.13,>=3.8.1; 0.1.3 Requires-Python <3.13,>=3.8.1; 0.2.0 Requires-Python <3.13,>=3.9; 0.2.0.dev1 Requires-Python <3.13,>=3.9
ERROR: Could not find a version that satisfies the requirement simsimd<4.0.0,>=3.6.3 (from langchain-pinecone) (from versions: 4.4.0, 5.0.0, 5.0.1, 5.1.0, 5.1.1, 5.1.2, 5.1.3, 5.1.4, 5.2.0, 5.2.1, 5.3.0, 5.4.0, 5.4.1, 5.4.2, 5.4.3, 5.4.4, 5.5.0, 5.5.1, 5.6.0, 5.6.1, 5.6.3, 5.6.4, 5.7.0, 5.7.1, 5.7.2, 5.7.3, 5.8.0, 5.9.0, 5.9.1, 5.9.2, 5.9.3, 5.9.4, 5.9.

# Clone a github repo locally

In [135]:
from git import GitCommandError, Repo
import os
import re
import requests
from pprint import pprint
from typing import Any, Dict, Generator, Tuple

In [231]:
SUPPORTED_EXTENSIONS = {'.py', '.js', '.tsx', '.jsx', '.ipynb', '.java',
                         '.cpp', '.ts', '.go', '.rs', '.vue', '.swift', '.c', '.h'}

IGNORED_DIRS = {
    'node_modules', 
    'venv', 
    'env', 
    'dist', 
    'build', 
    'vendor',
    '__pycache__', 
    'pacakge.json',
    'tsconfig.json'
}

In [243]:
import re
import os
import requests
from git import Repo, GitCommandError, InvalidGitRepositoryError

class GithubRepoData:
    def __init__(self, url: str):
        """
        Initializes the GithubRepoData object with repository details.

        Args:
            url (str): The GitHub repository URL.
            dir (str): The local directory to clone the repository into. Defaults to the current directory.
        """
        # Match the URL to extract the owner and repo name
        pattern = r"(?:https?://|git@)github\.com[:/](.+?)/(.+?)(?:/|\.git|$)"
        match = re.match(pattern, url)
        if not match:
            raise ValueError(f"Invalid GitHub URL: {url}")
        owner, repo = match.groups()

        # Fetch repository details from GitHub API
        api_url = f'https://api.github.com/repos/{owner}/{repo}'
        response = requests.get(api_url)
        if response.status_code == 200:
            github = response.json()
            visibility = github.get('visibility', 'unknown')

            if visibility == 'public':
                self.id = github.get('id')
                self.owner = github.get('owner').get('login')
                self.fullname = github.get('full_name')
                self.repo_name = github.get('name')
                self.description = github.get('description')
                self.events_url = github.get('events_url')
                # Set the directory path to the current directory if `dir` is not provided
                self.dir_path = os.path.join('repos', self.repo_name)
            else:
                raise ValueError(f'Repository {self.owner}/{self.repo_name} is not public.')
        elif response.status_code == 404:
            raise ValueError(f"Repository {self.owner}/{self.repo_name} not found.")
        elif response.status_code == 403:
            raise ValueError("GitHub API rate limit exceeded. Please try again later.")
        else:
            raise ValueError(f"Failed to fetch repository details for {self.owner}/{self.repo_name}. HTTP Status: {response.status_code}")

    def to_dict(self):
        return {
            "id": self.id,
            "owner": self.owner,
            "fullname": self.fullname,
            "repo_name": self.repo_name,
            "description": self.description,
            "events_url": self.events_url,
            "dir_path": self.dir_path,
            "url": f"https://github.com/{self.owner}/{self.repo_name}"
        }
    
    def exists(self) -> bool:
        """
        Checks if the repository is already cloned.

        Returns:
            bool: True if the repository is already cloned, False otherwise.
        """
        if os.path.exists(self.dir_path):
            try:
                # Check if the directory is a valid Git repository
                Repo(self.dir_path).git_dir
                print(f"The repository {self.repo_name} is already cloned at {self.dir_path}.")
                return True
            except InvalidGitRepositoryError:
                print(f"The directory {self.dir_path} exists but is not a valid Git repository.")
        return False

    def clone(self) -> bool:
        """
        Clones the repository into the specified local directory.

        Returns:
            bool: True if the cloning is successful.

        Raises:
            ValueError: If cloning fails.
        """
        if self.exists():
            print(f"Repository {self.repo_name} already cloned at {self.dir_path}.")
            return True

        # Create the local directory if it doesn't exist
        if not os.path.exists(self.dir_path):
            os.makedirs(self.dir_path, exist_ok=True)
        print(f'Cloning {self.repo_name} into {self.dir_path}')

        # Clone the repository
        try:
            clone_url = f"https://github.com/{self.fullname}.git"
            Repo.clone_from(clone_url, self.dir_path)
            print(f"Repository {self.repo_name} cloned successfully.")
            return True
        except GitCommandError as e:
            raise ValueError(f'Failed to clone {self.repo_name}: {e}')
        
    def read_file(self, filepath):
        """Reads the contents of a file in the repository."""
        try:
            with open(filepath, "r", encoding='utf-8') as f:
                contents = f.read()
            return contents
        except Exception as e:
            raise ValueError(f"Failed to read file {filepath}: {e}")
    
    def get_files(self) -> Generator[str, None, None]:
            """
            Recursively retrieves all files in `self.dir_name`, filtered by `supported_extensions` and skipping directories
            listed in `ignored_dirs`.

            Yields:
                str: Relative file paths for files that match the criteria.
            """
            for root, _, files in os.walk(self.dir_path):
                # Skip if current directory is in ignored directories
                if any(ignored_dir in root for ignored_dir in IGNORED_DIRS):
                    continue

                # Process each file in current directory
                for file in files:
                    file_path = os.path.join(root, file)
                    if os.path.splitext(file)[1] in SUPPORTED_EXTENSIONS:
                        content = self.read_file(file_path)

                        if content:
                            yield {
                                "path": file_path,
                                "url": None,
                                "content": content,
                            }


In [244]:
repo = GithubRepoData("https://github.com/CoderAgent/SecureAgent")
repo.clone()
repo.to_dict()
files = repo.get_files()
for file in files:
    print(file)

The repository SecureAgent is already cloned at repos/SecureAgent.
Repository SecureAgent already cloned at repos/SecureAgent.
{'path': 'repos/SecureAgent/src/app.ts', 'url': None, 'content': 'import { Octokit } from "@octokit/rest";\nimport { createNodeMiddleware } from "@octokit/webhooks";\nimport { WebhookEventMap } from "@octokit/webhooks-definitions/schema";\nimport * as http from "http";\nimport { App } from "octokit";\nimport { Review } from "./constants";\nimport { env } from "./env";\nimport { processPullRequest } from "./review-agent";\nimport { applyReview } from "./reviews";\n\n// This creates a new instance of the Octokit App class.\nconst reviewApp = new App({\n  appId: env.GITHUB_APP_ID,\n  privateKey: env.GITHUB_PRIVATE_KEY,\n  webhooks: {\n    secret: env.GITHUB_WEBHOOK_SECRET,\n  },\n});\n\nconst getChangesPerFile = async (payload: WebhookEventMap["pull_request"]) => {\n  try {\n    const octokit = await reviewApp.getInstallationOctokit(\n      payload.installation.id

In [171]:
response = requests.get('https://api.github.com/repos/CoderAgent/SecureAgent')
response
# response = requests.get(response['events_url'])
response = response.json()
response
# for r in response:
#     print(r)
# print(response["name"])
# print(response["full_name"])
# print(response["description"])
# print(response["private"])
# print(response["visibility"])
# print(response["clone_url"])


{'id': 719393996,
 'node_id': 'R_kgDOKuEUzA',
 'name': 'SecureAgent',
 'full_name': 'CoderAgent/SecureAgent',
 'private': False,
 'owner': {'login': 'CoderAgent',
  'id': 147278620,
  'node_id': 'O_kgDOCMdLHA',
  'avatar_url': 'https://avatars.githubusercontent.com/u/147278620?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/CoderAgent',
  'html_url': 'https://github.com/CoderAgent',
  'followers_url': 'https://api.github.com/users/CoderAgent/followers',
  'following_url': 'https://api.github.com/users/CoderAgent/following{/other_user}',
  'gists_url': 'https://api.github.com/users/CoderAgent/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/CoderAgent/starred{/owner}{/repo}',
  'subscriptions_url': 'https://api.github.com/users/CoderAgent/subscriptions',
  'organizations_url': 'https://api.github.com/users/CoderAgent/orgs',
  'repos_url': 'https://api.github.com/users/CoderAgent/repos',
  'events_url': 'https://api.github.com/users/CoderAgent/events{/pr

In [172]:
pattern = r"(?:https?://|git@)github\.com[:/](.+?)/(.+?)(?:/|\.git|$)"
match = re.match(pattern, "https://github.com/CoderAgent/SecureAgent")
match.groups()

('CoderAgent', 'SecureAgent')